# Gabarito — Exercícios de Machine Learning

Uma sugestão de resolução para cada exercício de `exercicios_ml.ipynb`. Não é a única forma correta — se você chegou num resultado parecido por outro caminho, também vale!

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(1)
n = 60

casas = pd.DataFrame({
    "area_m2": np.random.randint(40, 250, n),
    "quartos": np.random.randint(1, 5, n),
    "banheiros": np.random.randint(1, 4, n),
    "idade_imovel": np.random.randint(0, 40, n),
    "distancia_centro_km": np.round(np.random.uniform(0.5, 20, n), 1),
})

casas["preco"] = (
    casas["area_m2"] * 3500
    + casas["quartos"] * 15000
    + casas["banheiros"] * 8000
    - casas["idade_imovel"] * 1000
    - casas["distancia_centro_km"] * 2000
    + np.random.normal(0, 10000, n)
).round(2)

casas.loc[[3, 15, 30, 45], "idade_imovel"] = np.nan

casas.head()

## 1. Basic Data Exploration

In [ ]:
# 1.1
casas.describe()

In [ ]:
# 1.2
print(casas.shape)
print(casas.dtypes)

In [ ]:
# 1.3
casas.isnull().sum()

In [ ]:
# 1.4
casas_limpo = casas.dropna()
casas_limpo.shape

## 2. Your First Machine Learning Model

In [ ]:
# 2.1
y = casas_limpo["preco"]

In [ ]:
# 2.2
features = ["area_m2", "quartos", "banheiros", "idade_imovel", "distancia_centro_km"]
X = casas_limpo[features]

In [ ]:
# 2.3
from sklearn.tree import DecisionTreeRegressor

modelo = DecisionTreeRegressor(random_state=1)
modelo.fit(X, y)

In [ ]:
# 2.4
print("Previsões:", modelo.predict(X.head()))
print("Valores reais:", y.head().values)

## 3. Model Validation

In [ ]:
# 3.1
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X, y, random_state=1)

In [ ]:
# 3.2
modelo_val = DecisionTreeRegressor(random_state=1)
modelo_val.fit(X_train, y_train)
previsoes_val = modelo_val.predict(X_val)

In [ ]:
# 3.3
from sklearn.metrics import mean_absolute_error

mae = mean_absolute_error(y_val, previsoes_val)
mae

## 4. Underfitting and Overfitting

In [ ]:
# 4.1
def get_mae(max_leaf_nodes, X_train, X_val, y_train, y_val):
    modelo = DecisionTreeRegressor(max_leaf_nodes=max_leaf_nodes, random_state=1)
    modelo.fit(X_train, y_train)
    previsoes = modelo.predict(X_val)
    return mean_absolute_error(y_val, previsoes)

In [ ]:
# 4.2
candidatos = [5, 25, 50, 100, 250, 500]
maes = {n: get_mae(n, X_train, X_val, y_train, y_val) for n in candidatos}
print(maes)

melhor_max_leaf_nodes = min(maes, key=maes.get)
melhor_max_leaf_nodes

In [ ]:
# 4.3
modelo_final = DecisionTreeRegressor(max_leaf_nodes=melhor_max_leaf_nodes, random_state=1)
modelo_final.fit(X, y)

## 5. Random Forests

In [ ]:
# 5.1
from sklearn.ensemble import RandomForestRegressor

modelo_rf = RandomForestRegressor(random_state=1)
modelo_rf.fit(X_train, y_train)
previsoes_rf = modelo_rf.predict(X_val)
mae_rf = mean_absolute_error(y_val, previsoes_rf)
mae_rf

In [ ]:
# 5.2
print(f"MAE árvore de decisão (melhor): {maes[melhor_max_leaf_nodes]:.2f}")
print(f"MAE random forest: {mae_rf:.2f}")